In [61]:
# Imports
import mediapipe as mp
import pandas as pd
import os

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)

import cv2


I0000 00:00:1742653934.947552 10100699 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1


In [62]:
writepath = './process_files/pose.csv'
write_test_path = './process_files/test.csv'

os.makedirs(os.path.dirname(writepath), exist_ok=True)
mode = 'a' if os.path.exists(writepath) else 'w'

In [63]:
def process_images(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None
    
    # Resize the image to 640x640
    image_resized = cv2.resize(image, (640, 640))
    
    # Convert the resized image to RGB as MediaPipe works in RGB
    image_rgb = cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)
    
    # Process the image with MediaPipe Pose
    results = pose.process(image_rgb)
    
    if results.pose_landmarks:
        # Extract landmarks and return them as individual values
        landmarks = []
        for landmark in results.pose_world_landmarks.landmark:
            landmarks.extend([landmark.x, landmark.y, landmark.visibility])
        return landmarks
    return None

In [64]:
def save_to_csv(data, mode, writepath):
    # Dynamically create column names for landmarks (x, y, z, visibility for each landmark)
    num_landmarks = len(data[0]) // 3  # Each landmark has 4 values (x, y, z, visibility)
    columns = ['Label']  # Start with Label column

    # Create columns for each landmark's x, y, z, and visibility
    for i in range(num_landmarks):
        columns.append(f"landmark_{i}_x")
        columns.append(f"landmark_{i}_y")
        # columns.append(f"landmark_{i}_z")
        columns.append(f"landmark_{i}_visibility")
    
    # Create DataFrame with the correct column names
    df = pd.DataFrame(data, columns=columns)
    
    # Save to CSV file, write headers only if the file doesn't exist
    df.to_csv(writepath, mode=mode, header=not os.path.exists(writepath), index=False)



In [65]:
# Load Raw Data


def process_data(awake_path, sleep_path, writepath):
    data = []
    
    # Process Awake data
    for filename in os.listdir(awake_path):
        file_path = os.path.join(awake_path, filename)
        landmarks = process_images(file_path)
        if landmarks:
            data.append(['1'] + landmarks)  # Add label (Awake) and landmarks
    
    # Process Sleep data
    for filename in os.listdir(sleep_path):
        file_path = os.path.join(sleep_path, filename)
        landmarks = process_images(file_path)
        if landmarks:
            data.append(['0'] + landmarks)  # Add label (Sleep) and landmarks
    
    # Save the data to CSV
    save_to_csv(data, mode, writepath)

In [66]:
# Put Data into csv
awake_train_path = './raw_data/train/Awake'
sleep_train_path = './raw_data/train/Sleep'

awake_test_path = './raw_data/test/Awake'
sleep_test_path = './raw_data/test/Sleep'

process_data(awake_train_path, sleep_train_path, writepath)
process_data(awake_test_path, sleep_test_path, write_test_path)


W0000 00:00:1742653935.037018 10271680 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1742653935.053498 10271680 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
